# Lab 2: DDPG & SAC

**Course:** Deep Reinforcement Learning · **Framework:** Stable Baselines3 · **Estimated time:** ~60 min

> **Note on Spinning Up TF1.** The original course exercises reference Spinning Up's TensorFlow 1.x implementations, which require Python ≤ 3.7 and TF 1.15. Modern Colab runtimes use Python 3.10+, making those imports fail. This lab uses [Stable Baselines3](https://stable-baselines3.readthedocs.io) (SB3), which implements the same algorithms — DDPG, TD3, SAC — in a maintained codebase with an identical conceptual interface.

## Learning Objectives

- Run a baseline DDPG experiment on Pendulum-v1, interpret the training metrics, and evaluate the saved policy
- Deliberately destabilize DDPG by tuning learning rates and target smoothing; compare against the stable baseline
- Run a controlled SAC vs DDPG comparison on HalfCheetah-v4 across 3 seeds
- Conduct a SAC entropy coefficient (α) ablation across α ∈ {0.05, 0.1, 0.2, 0.5} on Hopper-v4
- Implement a custom Gym-compatible environment and train SAC on it, including a sparse reward variant
- Observe how value function quality affects policy gradient via a GAE-Lambda ablation on PPO (Problem Set 2.1)
- Diagnose a silent DDPG bug from degraded learning curves (Problem Set 2.2)

## Setup

In [ ]:
!pip install stable-baselines3[extra] gymnasium shimmy -q
!pip install gymnasium[mujoco] -q

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import DDPG, SAC, TD3, PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy

# Verify environments load
env = gym.make("Pendulum-v1")
obs, info = env.reset()
print(f"Pendulum-v1 obs shape: {obs.shape}, action shape: {env.action_space.shape}")
env.close()

## Exercise 1: Baseline DDPG on Pendulum-v1

Run a full DDPG baseline. Pendulum-v1 is a classic continuous control task — the agent must swing a pendulum upright and hold it there.

In [ ]:
import os

log_dir = "/tmp/ddpg-pendulum/"
os.makedirs(log_dir, exist_ok=True)

env = Monitor(gym.make("Pendulum-v1"), log_dir)

n_actions     = env.action_space.shape[-1]
action_noise  = NormalActionNoise(
    mean  = np.zeros(n_actions),
    sigma = 0.1 * np.ones(n_actions)
)

model = DDPG(
    "MlpPolicy", env,
    learning_rate  = 1e-3,
    buffer_size    = 1_000_000,
    batch_size     = 100,
    gamma          = 0.99,
    tau            = 0.005,          # equivalent to polyak = 0.995
    action_noise   = action_noise,
    learning_starts= 10_000,
    policy_kwargs  = dict(net_arch=[64, 64]),
    verbose        = 1,
    tensorboard_log= log_dir,
)
model.learn(total_timesteps=200_000)
model.save("/tmp/ddpg-pendulum/model")

In [ ]:
# Evaluate the saved policy
eval_env = gym.make("Pendulum-v1")
model    = DDPG.load("/tmp/ddpg-pendulum/model", env=eval_env)

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)
print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
eval_env.close()

In [ ]:
# Plot training curve from Monitor logs
results = load_results(log_dir)
x, y = ts2xy(results, 'timesteps')

# Smooth with a rolling window
window = 20
y_smooth = np.convolve(y, np.ones(window)/window, mode='valid')
x_smooth = x[window-1:]

plt.figure(figsize=(8, 4))
plt.plot(x, y, alpha=0.3, color='steelblue', label='Episode reward')
plt.plot(x_smooth, y_smooth, color='steelblue', label='Smoothed (window=20)')
plt.xlabel('Timesteps')
plt.ylabel('Episode Reward')
plt.title('DDPG on Pendulum-v1')
plt.legend()
plt.tight_layout()
plt.show()

**Tasks:**
1. Does the smoothed reward curve converge to near -200 (optimal) or remain much lower?
2. What does a non-converging Q-value (equivalent to `QVals` in Spinning Up) look like? Add a callback to track `mean_q_value`.
3. Try loading the model and running it in a rendered environment.

## Exercise 2: Observing DDPG Instability

DDPG is notoriously sensitive to hyperparameters. Create instability deliberately, then compare against the stable defaults.

In [ ]:
log_dir_stable   = "/tmp/ddpg-stable/"
log_dir_unstable = "/tmp/ddpg-unstable/"
os.makedirs(log_dir_stable,   exist_ok=True)
os.makedirs(log_dir_unstable, exist_ok=True)

# ── Stable baseline ──────────────────────────────────────────────
env_stable = Monitor(gym.make("HalfCheetah-v4"), log_dir_stable)
n_actions  = env_stable.action_space.shape[-1]

model_stable = DDPG(
    "MlpPolicy", env_stable,
    learning_rate  = 1e-3,
    tau            = 0.005,
    action_noise   = NormalActionNoise(np.zeros(n_actions), 0.1 * np.ones(n_actions)),
    learning_starts= 10_000,
    policy_kwargs  = dict(net_arch=[256, 256]),
    verbose        = 0,
)
model_stable.learn(total_timesteps=200_000)
env_stable.close()
print("Stable run complete.")

In [ ]:
# ── Deliberately unstable: 10x learning rate, reduced tau, minimal exploration ──
env_unstable = Monitor(gym.make("HalfCheetah-v4"), log_dir_unstable)

model_unstable = DDPG(
    "MlpPolicy", env_unstable,
    learning_rate  = 0.01,      # 10x default — unstable
    tau            = 0.01,      # less smoothing (higher tau = faster target update)
    action_noise   = NormalActionNoise(np.zeros(n_actions), 0.1 * np.ones(n_actions)),
    learning_starts= 1_000,     # much less random exploration
    policy_kwargs  = dict(net_arch=[256, 256]),
    verbose        = 0,
)
model_unstable.learn(total_timesteps=200_000)
env_unstable.close()
print("Unstable run complete.")

In [ ]:
# Plot both training curves side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, log_dir, label, color in [
    (axes[0], log_dir_stable,   "Stable (lr=1e-3, τ=0.005)", "steelblue"),
    (axes[1], log_dir_unstable, "Unstable (lr=0.01, τ=0.01)", "tomato"),
]:
    results = load_results(log_dir)
    x, y = ts2xy(results, 'timesteps')
    if len(x) > 20:
        y_s = np.convolve(y, np.ones(20)/20, mode='valid')
        ax.plot(x[19:], y_s, color=color, label=label)
        ax.plot(x, y, alpha=0.2, color=color)
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Episode Reward')
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

**Observations:**
- Does the unstable run's reward crash after an initial peak?
- Why does a high learning rate destabilize the critic (Q-network)?
- How does `tau` (target network update rate) relate to Spinning Up's `polyak` parameter? (`tau = 1 - polyak`)

## Exercise 3: SAC vs DDPG on HalfCheetah-v4

Compare SAC and DDPG head-to-head with matched compute budgets across 3 seeds each.

In [ ]:
import json
from stable_baselines3.common.callbacks import EvalCallback

def run_experiment(AlgoClass, algo_name, env_name, seed,
                   total_timesteps=400_000, algo_kwargs=None):
    log_dir  = f"/tmp/{algo_name}-{env_name.lower().split('-')[0]}-s{seed}/"
    eval_dir = f"/tmp/{algo_name}-{env_name.lower().split('-')[0]}-s{seed}-eval/"
    os.makedirs(log_dir,  exist_ok=True)
    os.makedirs(eval_dir, exist_ok=True)

    env      = Monitor(gym.make(env_name), log_dir)
    eval_env = Monitor(gym.make(env_name), eval_dir)

    kwargs = algo_kwargs or {}
    model  = AlgoClass("MlpPolicy", env, seed=seed,
                       policy_kwargs=dict(net_arch=[256,256]),
                       verbose=0, **kwargs)

    eval_cb = EvalCallback(eval_env, eval_freq=10_000, n_eval_episodes=5,
                           log_path=eval_dir, deterministic=True, verbose=0)
    model.learn(total_timesteps=total_timesteps, callback=eval_cb)
    env.close(); eval_env.close()
    print(f"{algo_name} seed={seed} done")
    return eval_dir

ddpg_kwargs = dict(
    learning_rate  = 1e-3,
    tau            = 0.005,
    action_noise   = NormalActionNoise(np.zeros(6), 0.1 * np.ones(6)),
    learning_starts= 10_000,
)
sac_kwargs = dict(learning_rate=1e-3, tau=0.005, ent_coef=0.2)

ddpg_dirs, sac_dirs = [], []
for seed in [0, 10, 20]:
    ddpg_dirs.append(run_experiment(DDPG, 'ddpg', 'HalfCheetah-v4', seed, algo_kwargs=ddpg_kwargs))
    sac_dirs.append( run_experiment(SAC,  'sac',  'HalfCheetah-v4', seed, algo_kwargs=sac_kwargs))

In [ ]:
def load_eval_results(eval_dir):
    path = os.path.join(eval_dir, 'evaluations.npz')
    data = np.load(path)
    timesteps = data['timesteps']
    results   = data['results'].mean(axis=1)   # mean over eval episodes
    return timesteps, results

fig, ax = plt.subplots(figsize=(9, 5))

for dirs, label, color in [(ddpg_dirs, 'DDPG', 'steelblue'), (sac_dirs, 'SAC', 'darkorange')]:
    all_x, all_y = [], []
    for d in dirs:
        x, y = load_eval_results(d)
        all_x.append(x); all_y.append(y)
    min_len = min(len(y) for y in all_y)
    x_arr   = all_x[0][:min_len]
    y_arr   = np.array([y[:min_len] for y in all_y])
    y_mean  = y_arr.mean(axis=0)
    y_std   = y_arr.std(axis=0)

    ax.plot(x_arr, y_mean, color=color, label=label, linewidth=2)
    ax.fill_between(x_arr, y_mean - y_std, y_mean + y_std, alpha=0.2, color=color)

ax.set_xlabel('Timesteps')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('SAC vs DDPG on HalfCheetah-v4 (3 seeds)')
ax.legend()
plt.tight_layout()
plt.show()

**Analysis:**
- Which algorithm achieves higher reward at 400k timesteps?
- Which has less variance across seeds?
- At approximately what timestep does SAC surpass DDPG's final performance?
- Why is SAC generally more stable? (Hint: entropy regularisation and dual critics)

## Exercise 4: SAC Entropy Coefficient Ablation

SAC's entropy coefficient α controls the exploration-exploitation tradeoff. Run an ablation across α ∈ {0.05, 0.1, 0.2, 0.5} on Hopper-v4.

In [ ]:
alpha_results = {}

for alpha in [0.05, 0.1, 0.2, 0.5]:
    eval_dir = f"/tmp/sac-hopper-alpha{alpha}/"
    os.makedirs(eval_dir, exist_ok=True)

    env      = gym.make("Hopper-v4")
    eval_env = Monitor(gym.make("Hopper-v4"), eval_dir)

    model = SAC("MlpPolicy", env,
                ent_coef      = alpha,
                learning_rate = 3e-4,
                policy_kwargs = dict(net_arch=[256, 256]),
                verbose       = 0)

    eval_cb = EvalCallback(eval_env, eval_freq=10_000, n_eval_episodes=5,
                           log_path=eval_dir, deterministic=True, verbose=0)
    model.learn(total_timesteps=400_000, callback=eval_cb)
    env.close(); eval_env.close()

    x, y = load_eval_results(eval_dir)
    alpha_results[alpha] = (x, y)
    print(f"alpha={alpha:.2f} | final mean reward: {y[-1]:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1a7abf', '#2ba02b', '#d65f07', '#9b3ab2']

for (alpha, (x, y)), color in zip(alpha_results.items(), colors):
    ax.plot(x, y, label=f'α={alpha}', color=color, linewidth=2)

ax.set_xlabel('Timesteps')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('SAC Entropy Coefficient Ablation on Hopper-v4')
ax.legend()
plt.tight_layout()
plt.show()

**Discussion:**
- How does very low α (0.05) behave vs. high α (0.5)?
- Which α converges fastest on Hopper? Does it also achieve the highest final reward?
- Why might a high α help in early training but hurt final performance?

> **Adaptive α:** SB3's SAC supports `ent_coef='auto'`, which automatically tunes α to maintain a target entropy level. Try it — does it outperform the best fixed α you found?

## Exercise 5: Custom Gymnasium Environment

Create a custom Gymnasium-compatible environment and train SAC on it. The task: a 2D point-mass must navigate to a randomly placed goal.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class SimpleReachEnv(gym.Env):
    """2D point-mass reaching task — dense reward (negative distance)."""
    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=-2, high=2, shape=(4,), dtype=np.float32
        )  # [agent_x, agent_y, goal_x, goal_y]
        self.action_space = spaces.Box(
            low=-1, high=1, shape=(2,), dtype=np.float32
        )  # [dx, dy]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pos  = self.np_random.uniform(-1, 1, 2).astype(np.float32)
        self.goal = self.np_random.uniform(-1, 1, 2).astype(np.float32)
        return np.concatenate([self.pos, self.goal]), {}

    def step(self, action):
        self.pos  = np.clip(self.pos + 0.1 * action, -2, 2).astype(np.float32)
        dist      = float(np.linalg.norm(self.pos - self.goal))
        reward    = -dist
        terminated= dist < 0.05
        return np.concatenate([self.pos, self.goal]), reward, terminated, False, {}


# Register and train
gym.register(id='SimpleReach-v0', entry_point=SimpleReachEnv, max_episode_steps=50)

env      = gym.make('SimpleReach-v0')
eval_env = Monitor(gym.make('SimpleReach-v0'), "/tmp/sac-reach/")
os.makedirs("/tmp/sac-reach/", exist_ok=True)

model = SAC("MlpPolicy", env, ent_coef=0.1, policy_kwargs=dict(net_arch=[64,64]), verbose=1)
eval_cb = EvalCallback(eval_env, eval_freq=2_000, n_eval_episodes=20,
                       log_path="/tmp/sac-reach/", deterministic=True, verbose=0)
model.learn(total_timesteps=100_000, callback=eval_cb)
env.close(); eval_env.close()

In [ ]:
# Sparse reward variant — does SAC still learn?
class SimpleReachSparseEnv(SimpleReachEnv):
    def step(self, action):
        obs, _, terminated, truncated, info = super().step(action)
        dist   = float(np.linalg.norm(self.pos - self.goal))
        reward = 1.0 if dist < 0.05 else 0.0
        return obs, reward, terminated, truncated, info

gym.register(id='SimpleReachSparse-v0',
             entry_point=SimpleReachSparseEnv, max_episode_steps=50)

env_sp      = gym.make('SimpleReachSparse-v0')
eval_env_sp = Monitor(gym.make('SimpleReachSparse-v0'), "/tmp/sac-reach-sparse/")
os.makedirs("/tmp/sac-reach-sparse/", exist_ok=True)

model_sp = SAC("MlpPolicy", env_sp, ent_coef=0.2,
               policy_kwargs=dict(net_arch=[64,64]), verbose=0)
eval_cb_sp = EvalCallback(eval_env_sp, eval_freq=2_000, n_eval_episodes=20,
                          log_path="/tmp/sac-reach-sparse/", deterministic=True, verbose=0)
model_sp.learn(total_timesteps=200_000, callback=eval_cb_sp)
env_sp.close(); eval_env_sp.close()

**Tasks:**
- Does the dense reward variant achieve near-zero episode return (agent reaches goal)?
- Does the sparse variant converge? How many more timesteps does it require?
- What modifications to the environment or algorithm would help with sparse rewards?

## Problem Set 2.1: Value Function Quality in Policy Gradient

**Background.** GAE-Lambda uses V^π as a baseline to estimate advantages. Poor value function fitting → high-variance advantage estimates → unreliable policy updates.

We replicate the Spinning Up Exercise 2.1 concept using SB3's PPO, comparing `n_epochs=1` (minimal value fitting) against `n_epochs=10` (proper fitting) on Hopper-v4.

In [ ]:
ps_results = {}

for n_epochs, label in [(1, 'n_epochs=1 (poor VF)'), (10, 'n_epochs=10 (proper VF)')]:
    eval_dir = f"/tmp/ppo-hopper-vf{n_epochs}/"
    os.makedirs(eval_dir, exist_ok=True)

    results_per_seed = []
    for seed in [0, 10, 20]:
        env      = gym.make("Hopper-v4")
        eval_env = Monitor(gym.make("Hopper-v4"), eval_dir)

        model = PPO("MlpPolicy", env,
                    n_steps   = 4096,
                    n_epochs  = n_epochs,   # gradient steps on VF per rollout
                    gae_lambda= 0.97,
                    gamma     = 0.99,
                    seed      = seed,
                    policy_kwargs = dict(net_arch=dict(pi=[64,64], vf=[64,64])),
                    verbose   = 0)

        eval_cb = EvalCallback(eval_env, eval_freq=20_000, n_eval_episodes=5,
                               log_path=eval_dir + f"seed{seed}/",
                               deterministic=True, verbose=0)
        model.learn(total_timesteps=1_000_000, callback=eval_cb)
        env.close(); eval_env.close()

        x, y = load_eval_results(eval_dir + f"seed{seed}/")
        results_per_seed.append((x, y))
        print(f"  {label} seed={seed} | final: {y[-1]:.0f}")

    ps_results[label] = results_per_seed

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue', 'darkorange']

for (label, seeds_data), color in zip(ps_results.items(), colors):
    min_len = min(len(y) for _, y in seeds_data)
    x_arr   = seeds_data[0][0][:min_len]
    y_arr   = np.array([y[:min_len] for _, y in seeds_data])
    y_mean  = y_arr.mean(axis=0)
    y_std   = y_arr.std(axis=0)

    ax.plot(x_arr, y_mean, label=label, color=color, linewidth=2)
    ax.fill_between(x_arr, y_mean - y_std, y_mean + y_std, alpha=0.2, color=color)

ax.set_xlabel('Timesteps')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('Value Function Fitting Quality in PPO (Hopper-v4)')
ax.legend()
plt.tight_layout()
plt.show()

**Analysis questions:**
- What is the final reward gap between `n_epochs=1` and `n_epochs=10`?
- Does `n_epochs=1` make *any* progress, or flatline?
- Why does poor value fitting hurt policy gradient so much? (Hint: high-variance advantage estimates inflate the gradient noise.)

## Problem Set 2.2: Silent Bug in DDPG

**Key lesson:** RL failures are frequently *silent* — code runs without errors, but the agent never learns.

The bug below is adapted from Spinning Up Exercise 2.2. A DDPG actor-critic is constructed incorrectly so that the actor and critic *share weights*. The critic loss still decreases, the actor loss still propagates gradients — but the agent fails to learn a useful policy.

**Your task:** Before reading the explanation, look at `build_buggy_ac()`, identify the bug, and explain how it breaks the computation graph.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal
import gymnasium as gym
import numpy as np
from collections import deque
import random

# ── Minimal DDPG implementation ───────────────────────────────────

def mlp(sizes, activation=nn.ReLU):
    layers = []
    for i in range(len(sizes)-1):
        layers += [nn.Linear(sizes[i], sizes[i+1])]
        if i < len(sizes)-2:
            layers += [activation()]
    return nn.Sequential(*layers)

class DDPGActor(nn.Module):
    def __init__(self, obs_dim, act_dim, act_limit):
        super().__init__()
        self.net = mlp([obs_dim, 256, 256, act_dim])
        self.act_limit = act_limit
    def forward(self, obs):
        return self.act_limit * torch.tanh(self.net(obs))

class DDPGCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = mlp([obs_dim + act_dim, 256, 256, 1])
    def forward(self, obs, act):
        return self.net(torch.cat([obs, act], dim=-1)).squeeze(-1)

# ── Correct actor-critic construction ────────────────────────────
def build_correct_ac(obs_dim, act_dim, act_limit):
    actor  = DDPGActor(obs_dim, act_dim, act_limit)
    critic = DDPGCritic(obs_dim, act_dim)
    # Separate networks — correct
    return actor, critic

# ── BUGGY actor-critic construction ──────────────────────────────
def build_buggy_ac(obs_dim, act_dim, act_limit):
    actor  = DDPGActor(obs_dim, act_dim, act_limit)
    # BUG: critic reuses the actor's network layers instead of having its own
    critic = DDPGCritic(obs_dim, act_dim)
    critic.net = actor.net   # <-- shared weights!
    return actor, critic

In [ ]:
class ReplayBuffer:
    def __init__(self, size=100_000):
        self.buf = deque(maxlen=size)
    def push(self, *args):
        self.buf.append(args)
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        return [torch.as_tensor(np.array(x), dtype=torch.float32)
                for x in zip(*batch)]
    def __len__(self):
        return len(self.buf)

def train_ddpg(build_ac_fn, env_name='Pendulum-v1',
               total_steps=50_000, start_steps=1000, batch_size=100,
               gamma=0.99, tau=0.005, lr=1e-3, act_noise=0.1):
    env = gym.make(env_name)
    obs_dim   = env.observation_space.shape[0]
    act_dim   = env.action_space.shape[0]
    act_limit = float(env.action_space.high[0])

    actor,  critic  = build_ac_fn(obs_dim, act_dim, act_limit)
    actor_t, critic_t = build_ac_fn(obs_dim, act_dim, act_limit)
    actor_t.load_state_dict(actor.state_dict())
    critic_t.load_state_dict(critic.state_dict())
    for p in list(actor_t.parameters()) + list(critic_t.parameters()):
        p.requires_grad_(False)

    pi_opt = torch.optim.Adam(actor.parameters(),  lr=lr)
    q_opt  = torch.optim.Adam(critic.parameters(), lr=lr)
    buf    = ReplayBuffer()

    obs, _ = env.reset()
    ep_ret, ep_rets = 0, []

    for t in range(total_steps):
        if t < start_steps:
            act = env.action_space.sample()
        else:
            with torch.no_grad():
                act = actor(torch.as_tensor(obs, dtype=torch.float32)).numpy()
                act = np.clip(act + act_noise * np.random.randn(act_dim), -act_limit, act_limit)

        obs2, rew, term, trunc, _ = env.step(act)
        done = term or trunc
        buf.push(obs, act, rew, obs2, float(done))
        obs   = obs2 if not done else env.reset()[0]
        ep_ret += rew

        if done:
            ep_rets.append(ep_ret)
            ep_ret = 0

        if len(buf) >= batch_size:
            o, a, r, o2, d = buf.sample(batch_size)
            with torch.no_grad():
                backup = r + gamma * (1 - d) * critic_t(o2, actor_t(o2))
            q_loss = ((critic(o, a) - backup)**2).mean()
            q_opt.zero_grad(); q_loss.backward(); q_opt.step()

            pi_loss = -critic(o, actor(o)).mean()
            pi_opt.zero_grad(); pi_loss.backward(); pi_opt.step()

            for p, pt in zip(actor.parameters(),  actor_t.parameters()):
                pt.data.mul_(1 - tau).add_(tau * p.data)
            for p, pt in zip(critic.parameters(), critic_t.parameters()):
                pt.data.mul_(1 - tau).add_(tau * p.data)

    env.close()
    return ep_rets

print("Training CORRECT DDPG...")
correct_rets = train_ddpg(build_correct_ac)
print(f"  Final 10 ep avg: {np.mean(correct_rets[-10:]):.1f}")

print("\nTraining BUGGY DDPG...")
buggy_rets = train_ddpg(build_buggy_ac)
print(f"  Final 10 ep avg: {np.mean(buggy_rets[-10:]):.1f}")

In [ ]:
# Compare learning curves
window = 10
def smooth(x, w):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(8, 4))
plt.plot(smooth(correct_rets, window), color='steelblue', label='Correct DDPG')
plt.plot(smooth(buggy_rets,   window), color='tomato',    label='Buggy DDPG (shared weights)')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.title('Correct vs Buggy DDPG on Pendulum-v1')
plt.legend()
plt.tight_layout()
plt.show()

**Diagnosis questions:**
1. What is the performance gap between the correct and buggy implementations?
2. Exactly which line in `build_buggy_ac()` introduces the bug?
3. How does sharing weights between actor and critic break the DDPG computation graph? Think about what happens when the critic loss is backpropagated through the shared network.
4. **Bonus:** Does the buggy DDPG make *any* learning progress, or does it flatline completely? Why might some hyperparameter settings partially hide the bug?